# SixR3C (EN ISO 13790) – Forward Pass Equations (with $y=[T_i,T_h,T_m]$ and $B=[0,1,0]$)

## 1) Shapes and notation
- State (per batch): $y=[T_i,\;T_h,\;T_m]\in\mathbb{R}^3$
- Matrices:
  - $A\in\mathbb{R}^{3\times 3}$ (state dynamics)
  - $B\in\mathbb{R}^{1\times 3}=\begin{bmatrix}0&1&0\end{bmatrix}$ (heat input to $T_h$)
  - $E\in\mathbb{R}^{3\times 3}$ (disturbance gains)
  - $\text{inv\_C}=\begin{bmatrix}1/C_{\text{air}},\;1/C_h,\;1/C_m\end{bmatrix}$

- Inputs at time $t$:
  - $\Phi_{HC}(t)\in\mathbb{R}$
  - Disturbance vector $d=\begin{bmatrix}T_a(t),\;\Phi_{sol}(t),\;1\end{bmatrix}\in\mathbb{R}^3$

The continuous-time ODE computed by the code is
$$
\dot y \;=\; \text{inv\_C}\;\odot\;\Big( yA^\top \;+\; \Phi_{HC}\,B \;+\; d\,E^\top \Big),
$$
where $\odot$ is element-wise multiplication with the 3-vector $\text{inv\_C}$.

---

## 2) The matrices
Let
$$
\begin{aligned}
X &= H_{is}+H_w+H_{ms}, \\
Hms\_X &= \tfrac{H_{ms}}{X}, \quad His\_X = \tfrac{H_{is}}{X}, \\
His^2\_X &= \tfrac{H_{is}^2}{X}, \quad Hms^2\_X = \tfrac{H_{ms}^2}{X}, \\
HisHms\_X &= \tfrac{H_{is}H_{ms}}{X}, \quad HisHw\_X = \tfrac{H_{is}H_w}{X}, \\
A_t &= 4.5A_f,\quad A_m=3.5A_f,\quad C_m=c_fA_f,\quad \text{AmAt}=\tfrac{A_m}{A_t},\\
Y &= 1 - \text{AmAt} - \tfrac{H_w}{9.1\,A_t}.
\end{aligned}
$$

Then
$$
A=\begin{bmatrix}
His^2\_X - (H_{ve}+H_{is}+H_{ih}) & H_{ih} & HisHms\_X\\
H_{ih} & -H_{ih} & 0\\
HisHms\_X & 0 & Hms^2\_X - (H_{em}+H_{ms})
\end{bmatrix},
\quad
B=\begin{bmatrix}0&1&0\end{bmatrix},
$$

$$
E=\begin{bmatrix}
H_{ve}+HisHw\_X & His\_X\,Y\,A_w & \big(His\_X\,Y+1\big)\tfrac{1}{2}\Phi_{int}\\
0 & 0 & 0\\
H_{em}+\tfrac{H_{ms}H_w}{X} & \big(Hms\_X\,Y+\text{AmAt}\big)A_w & \big(Hms\_X\,Y+\text{AmAt}\big)\tfrac{1}{2}\Phi_{int}
\end{bmatrix},
\quad
\text{inv\_C}=\begin{bmatrix}\tfrac{1}{C_{\text{air}}},\;\tfrac{1}{C_h},\;\tfrac{1}{C_m}\end{bmatrix}.
$$

---

## 3) Matrix products in the forward pass

### (a) State term: $yA^\top$
$$
yA^\top =
\begin{bmatrix}
T_i & T_h & T_m
\end{bmatrix}
\begin{bmatrix}
A_{11} & A_{21} & A_{31}\\
A_{12} & A_{22} & A_{32}\\
A_{13} & A_{23} & A_{33}
\end{bmatrix}
=
\begin{bmatrix}
T_iA_{11}+T_hA_{12}+T_mA_{13},\;
T_iA_{21}+T_hA_{22}+T_mA_{23},\;
T_iA_{31}+T_hA_{32}+T_mA_{33}
\end{bmatrix}.
$$

### (b) Control term: $\Phi_{HC} B$
$$
\Phi_{HC}B=\begin{bmatrix}0,\;\Phi_{HC},\;0\end{bmatrix}.
$$

### (c) Disturbance term: $dE^\top$
$$
dE^\top
=
\begin{bmatrix}
T_a & \Phi_{sol} & 1
\end{bmatrix}
E^\top
=
\begin{bmatrix}
T_aE_{11}+\Phi_{sol}E_{12}+E_{13},\;
T_aE_{21}+\Phi_{sol}E_{22}+E_{23},\;
T_aE_{31}+\Phi_{sol}E_{32}+E_{33}
\end{bmatrix}.
$$

---

## 4) Component-wise ODEs

### Indoor air node $T_i$:
$$
\dot T_i
=
\frac{1}{C_{\text{air}}}\Big[
\big(His^2\_X-(H_{ve}+H_{is}+H_{ih})\big) T_i
+ H_{ih} T_h
+ HisHms\_X\,T_m
+ (H_{ve}+HisHw\_X)T_a
+ (His\_X Y A_w)\,\Phi_{sol}
+ (His\_X Y +1)\tfrac{1}{2}\Phi_{int}
\Big].
$$

### Heating system node $T_h$:
$$
\dot T_h
=
\frac{1}{C_{h}}\Big[
H_{ih}T_i
- H_{ih}T_h
+ \Phi_{HC}
\Big].
$$

### Thermal mass node $T_m$:
$$
\dot T_m
=
\frac{1}{C_{m}}\Big[
HisHms\_X\,T_i
+ \big(Hms^2\_X-(H_{em}+H_{ms})\big)T_m
+ \Big(H_{em}+\tfrac{H_{ms}H_w}{X}\Big)T_a
+ \big(Hms\_X Y + \text{AmAt}\big)A_w\,\Phi_{sol}
+ \big(Hms\_X Y + \text{AmAt}\big)\tfrac{1}{2}\Phi_{int}
\Big].
$$

---

## 5) Quick implementation checks
- $B=[0,1,0]$ now routes $\Phi_{HC}$ to the $T_h$ equation (as in the equations above).
- $\text{inv\_C}=[1/C_{\text{air}},\,1/C_h,\,1/C_m]$ matches the state order $[T_i,T_h,T_m]$.


<!-- <img src="../../Literature/RC-models-structures/TiTeTh-RC-model.pn" 
     alt="TiTeTh thermal model structure" 
     style="max-width:50%; height:auto;"> -->

# 6R3C RC Network

## Components (states and capacities)
- **Indoor air node**: $T_{\text{air}}$ with thermal capacitance $C_{\text{air}}$
- **Thermal mass node**: $T_m$ (e.g., internal mass/envelope lump) with capacitance $C_m$
- **Heating system node**: $T_h$ with capacitance $C_h$

## Thermal conductances (W/K)
Using EN ISO 13790 style:
- $H_{\text{is}} = A_t \cdot 3.45$ — internal surface to air
- $H_{\text{ms}} = A_m \cdot 9.1$ — mass surface to air
- $H_w$ — ventilation/infiltration to ambient from air node
- $H_{\text{ve}}$ — mechanical ventilation to ambient from air node
- $H_{\text{em}}$ — mass/envelope to ambient from mass node
- $H_{\text{ih}}$ — coupling between heating system node and indoor air node

where:
- $A_t = 4.5\,A_f$  
- $A_m = 3.5\,A_f$  
- $X = H_{\text{is}} + H_w + H_{\text{ms}}$

## Inputs
- **Ambient temperature**: $T_a$
- **Internal gains**: $\Phi_{\text{int}}$
- **Solar irradiance**: $\Phi_{\text{sol}}$ (W/m$^2$), acting through window area $A_w$
- **Heating power into heating node**: $\Phi_{\text{HC}}$ (W)

## Gain splitting (ISO 13790)
$
\Phi_{\text{ia}} = 0.5\,\Phi_{\text{int}}, \quad
\Phi_{\text{ia,sol}} = \Phi_{\text{ia}} + \Phi_{\text{sol}}\,A_w
$

$
\Phi_m  = \left(\frac{A_m}{A_t}\right)\Phi_{\text{ia,sol}}, \quad
\Phi_{st} = \left(1 - \frac{A_m}{A_t} - \frac{H_w}{9.1\,A_t}\right)\Phi_{\text{ia,sol}}
$

## Heating–air coupling flow
$
Q_{\text{ih}} = H_{\text{ih}}\,(T_h - T_{\text{air}})
$

---

## Differential equations

### 1) Indoor air node $T_{\text{air}}$
$$
\begin{aligned}
C_{\text{air}}\,\frac{dT_{\text{air}}}{dt}
&= \left(\frac{H_{\text{is}}^2}{X} - (H_{\text{ve}} + H_{\text{is}})\right) T_{\text{air}} \\
&\quad + \frac{H_{\text{is}}H_{\text{ms}}}{X}\,T_m
+ \left(H_{\text{ve}} + \frac{H_{\text{is}}H_w}{X}\right) T_a \\
&\quad + \frac{H_{\text{is}}}{X}\,\Phi_{st}
+ \Phi_{\text{ia}}
+ Q_{\text{ih}}
\end{aligned}
$$

### 2) Thermal mass node $T_m$
$$
\begin{aligned}
C_m\,\frac{dT_m}{dt}
&= \left(\frac{H_{\text{ms}}^2}{X} - (H_{\text{em}} + H_{\text{ms}})\right) T_m \\
&\quad + \frac{H_{\text{is}}H_{\text{ms}}}{X}\,T_{\text{air}}
+ \left(H_{\text{em}} + \frac{H_{\text{ms}}H_w}{X}\right) T_a \\
&\quad + \frac{H_{\text{ms}}}{X}\,\Phi_{st}
+ \Phi_m
\end{aligned}
$$

### 3) Heating system node $T_h$
$$
C_h\,\frac{dT_h}{dt} = -Q_{\text{ih}} + \Phi_{\text{HC}}
= H_{\text{ih}}\,(T_{\text{air}} - T_h) + \Phi_{\text{HC}}
$$

---

## Summary of ODEs
$$
\begin{aligned}
C_{\text{air}}\,\dot T_{\text{air}} &= \left(\tfrac{H_{\text{is}}^2}{X} - (H_{\text{ve}} + H_{\text{is}})\right)T_{\text{air}}
+ \tfrac{H_{\text{is}}H_{\text{ms}}}{X}T_m
+ \left(H_{\text{ve}} + \tfrac{H_{\text{is}}H_w}{X}\right)T_a
+ \tfrac{H_{\text{is}}}{X}\Phi_{st}
+ \Phi_{\text{ia}} + Q_{\text{ih}}
\$4pt]
C_m\,\dot T_m &= \left(\tfrac{H_{\text{ms}}^2}{X} - (H_{\text{em}} + H_{\text{ms}})\right)T_m
+ \tfrac{H_{\text{is}}H_{\text{ms}}}{X}T_{\text{air}}
+ \left(H_{\text{em}} + \tfrac{H_{\text{ms}}H_w}{X}\right)T_a
+ \tfrac{H_{\text{ms}}}{X}\Phi_{st}
+ \Phi_m
\$4pt]
C_h\,\dot T_h &= H_{\text{ih}}(T_{\text{air}} - T_h) + \Phi_{\text{HC}}
\end{aligned}
$$